# GraphFlow (Workflows)

In this section We’ll learn how to create an multi-agent workflow using `GraphFlow`, or simply “flow” for short. It uses structured execution and precisely controls how agents interact to accomplish a task.<br><br>
We’ll first show you how to create and run a flow. We’ll then explain how to observe and debug flow behavior, and discuss important operations for managing execution.<br><br>
AutoGen AgentChat provides a team for directed graph execution:<br>
- **GraphFlow**: A team that follows a `DiGraph` to control the execution flow between agents. Supports sequential, parallel, conditional, and looping behaviors.
<br><br>

***Reference URL:-***
https://microsoft.github.io/autogen/stable//user-guide/agentchat-user-guide/graph-flow.html#sequential-flow

### GraphFlow

In [23]:
import asyncio
from autogen_agentchat.agents import AssistantAgent,UserProxyAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.conditions import TextMentionTermination

In [22]:
# Load API key

import os
from dotenv import load_dotenv

load_dotenv()
api_key= os.getenv('OPENAI_API_KEY')

In [9]:
# Model client

model_client= OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)

model_client

### DigraphBuilder GraphFlow

In [3]:
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow

### Sequential Flow
We will begin by creating a simple workflow where a writer drafts a paragraph and a reviewer provides feedback. This graph terminates after the reviewer comments on the writer. <br>
Note, the flow automatically computes all the source and leaf nodes of the graph and the execution starts at all the source nodes in the graph and completes execution when no nodes are left to execute.

In [10]:
writer= AssistantAgent(
    name="Writer",
    description="A writer agent that generates text based on user input.",
    model_client=model_client,
    system_message="You are a creative writer. Please write a story based on the user's input.",
)

reviewer= AssistantAgent(
    name="Reviewer",
    description="A reviewer agent that provides feedback on the text generated by the writer.",
    model_client=model_client,
    system_message="You are a reviewer. Please provide feedback on the text generated by the writer.",
)

In [11]:
# Build the Graph

builder= DiGraphBuilder()
builder.add_node(writer).add_node(reviewer)
builder.add_edge(writer, reviewer)

graph= builder.build()

In [12]:
# Visualize The Graph

graph

DiGraph(nodes={'Writer': DiGraphNode(name='Writer', edges=[DiGraphEdge(target='Reviewer', condition=None, condition_function=None, activation_group='Reviewer', activation_condition='all')], activation='all'), 'Reviewer': DiGraphNode(name='Reviewer', edges=[], activation='all')}, default_start_node=None)

In [ ]:
DiGraph(nodes={
    
    'Writer': DiGraphNode(name='Writer', edges=[DiGraphEdge(target='Reviewer', condition=None, condition_function=None, activation_group='Reviewer', activation_condition='all')], activation='all'), 
    
    'Reviewer': DiGraphNode(name='Reviewer', edges=[], activation='all')
    }, 
    default_start_node=None)

In [13]:
# Define Teams

team= GraphFlow([writer,reviewer], graph)
team

In [14]:
stream= team.run_stream(task="Write a good poem about India in less than 30 words.")

async for event in stream:
    print(event)

id='2558db42-b415-4a80-88b1-2a38acfc5e8d' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 7, 47, 34, 63388, tzinfo=datetime.timezone.utc) content='Write a good poem about India in less than 30 words.' type='TextMessage'
id='b9ca8079-011e-4572-a130-3d616963e387' source='Writer' models_usage=RequestUsage(prompt_tokens=41, completion_tokens=41) metadata={} created_at=datetime.datetime(2025, 7, 19, 7, 47, 36, 327613, tzinfo=datetime.timezone.utc) content='Land of colors, diverse and grand,  \nRivers caress the fertile sand.  \nMountains whisper ancient tales,  \nIndia, where unity prevails.  \nIn vibrant dance, hearts expand.  ' type='TextMessage'
id='83283b4a-5d03-4177-864c-942d205d5484' source='Reviewer' models_usage=RequestUsage(prompt_tokens=88, completion_tokens=130) metadata={} created_at=datetime.datetime(2025, 7, 19, 7, 47, 43, 857587, tzinfo=datetime.timezone.utc) content='This is a beautifully succinct poem that captures the essence of India 

##### Structured Output

In [ ]:
source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 7, 47, 34, 63388, tzinfo=datetime.timezone.utc) 
content='Write a good poem about India in less than 30 words.' type='TextMessage'

source='Writer' models_usage=RequestUsage(prompt_tokens=41, completion_tokens=41) metadata={} created_at=datetime.datetime(2025, 7, 19, 7, 47, 36, 327613, tzinfo=datetime.timezone.utc) 
content='Land of colors, diverse and grand,  \nRivers caress the fertile sand.  \nMountains whisper ancient tales,  \nIndia, where unity prevails.  \nIn vibrant dance, hearts expand.  ' type='TextMessage'

source='Reviewer' models_usage=RequestUsage(prompt_tokens=88, completion_tokens=130) metadata={} created_at=datetime.datetime(2025, 7, 19, 7, 47, 43, 857587, tzinfo=datetime.timezone.utc) 
content='This is a beautifully succinct poem that captures the essence of India in just a few lines. The imagery of diverse landscapes and cultural richness is effectively conveyed. The line "Mountains whisper ancient tales" brings a mystical touch, while "unity prevails" highlights India\'s core value of unity in diversity. The poem maintains a rhythm that complements its theme. Overall, it\'s a well-crafted piece that evokes both visual and emotional resonance. \n\nFor improvement, you might consider adding a specific reference to a cultural aspect, like a festival or a landmark, to further anchor the reader in India\'s unique identity. But for the constraints given, you have done an excellent job.' type='TextMessage'

source='DiGraphStopAgent' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 7, 47, 43, 859591, tzinfo=datetime.timezone.utc) 
content='Digraph execution is complete' type='StopMessage'

messages=[TextMessage(id='2558db42-b415-4a80-88b1-2a38acfc5e8d', 
                      
                      source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 7, 47, 34, 63388, tzinfo=datetime.timezone.utc), 
                      content='Write a good poem about India in less than 30 words.', type='TextMessage'), TextMessage(id='b9ca8079-011e-4572-a130-3d616963e387', 
                                                                                                                       
                      source='Writer', models_usage=RequestUsage(prompt_tokens=41, completion_tokens=41), metadata={}, created_at=datetime.datetime(2025, 7, 19, 7, 47, 36, 327613, tzinfo=datetime.timezone.utc), 
                      content='Land of colors, diverse and grand,  \nRivers caress the fertile sand.  \nMountains whisper ancient tales,  \nIndia, where unity prevails.  \nIn vibrant dance, hearts expand.  ', type='TextMessage'), TextMessage(id='83283b4a-5d03-4177-864c-942d205d5484', 
                                                                                                                                                                                                                                                  
                      source='Reviewer', models_usage=RequestUsage(prompt_tokens=88, completion_tokens=130), metadata={}, created_at=datetime.datetime(2025, 7, 19, 7, 47, 43, 857587, tzinfo=datetime.timezone.utc), 
                      content='This is a beautifully succinct poem that captures the essence of India in just a few lines. The imagery of diverse landscapes and cultural richness is effectively conveyed. The line "Mountains whisper ancient tales" brings a mystical touch, while "unity prevails" highlights India\'s core value of unity in diversity. The poem maintains a rhythm that complements its theme. Overall, it\'s a well-crafted piece that evokes both visual and emotional resonance. \n\nFor improvement, you might consider adding a specific reference to a cultural aspect, like a festival or a landmark, to further anchor the reader in India\'s unique identity. But for the constraints given, you have done an excellent job.', type='TextMessage'), 
                      
                      StopMessage(id='e4594e5e-6988-4349-843a-ada7802eedd9', source='DiGraphStopAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 7, 47, 43, 859591, tzinfo=datetime.timezone.utc), content='Digraph execution is complete', type='StopMessage')] 

stop_reason='Stop message received'

#### Example 1.

**`Sequential Flow: A → B → C`**

In [43]:
import asyncio

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow
from autogen_ext.models.openai import OpenAIChatCompletionClient


# async def main():
# Initialize agents with OpenAI model clients.
model_client = OpenAIChatCompletionClient(model="gpt-4.1-nano")
agent_a = AssistantAgent("A", model_client=model_client, system_message="You are a helpful assistant.")
agent_b = AssistantAgent("B", model_client=model_client, system_message="Translate input to Chinese.")
agent_c = AssistantAgent("C", model_client=model_client, system_message="Translate input to English.")

# Create a directed graph with sequential flow A -> B -> C.
builder = DiGraphBuilder()
builder.add_node(agent_a).add_node(agent_b).add_node(agent_c)
builder.add_edge(agent_a, agent_b).add_edge(agent_b, agent_c)
graph = builder.build()

# Create a GraphFlow team with the directed graph.
team = GraphFlow(
    participants=[agent_a, agent_b, agent_c],
    graph=graph,
    termination_condition=MaxMessageTermination(5),
)

# Run the team and print the events.
async for event in team.run_stream(task="Write a short story about a cat."):
    print(event)


# asyncio.run(main())


id='d8b55060-b5a3-423c-8412-687ed94ff616' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 55, 31, 332458, tzinfo=datetime.timezone.utc) content='Write a short story about a cat.' type='TextMessage'
id='7d10d8f5-b5f5-4e56-b424-be4f2b3f945a' source='A' models_usage=RequestUsage(prompt_tokens=26, completion_tokens=184) metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 55, 33, 122704, tzinfo=datetime.timezone.utc) content="Once upon a time, in a small cozy village, there lived a curious cat named Whiskers. With fur as soft as clouds and eyes as bright as emeralds, Whiskers loved exploring every corner of the neighborhood. Every morning, she wandered through gardens, chased butterflies, and napped in sunny spots on windowsills.\n\nOne day, while roaming near a tall oak tree, Whiskers discovered a tiny, shivering bird trapped beneath some fallen leaves. Without hesitation, she gently pawed at the debris, carefully freeing the tiny creature. The

##### Structured Output

**`Sequential Flow: A → B → C`**

In [ ]:
source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 49, 21, 301116, tzinfo=datetime.timezone.utc) 
content='Write a short story about a cat.' type='TextMessage'

source='A' models_usage=RequestUsage(prompt_tokens=26, completion_tokens=233) metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 49, 22, 861842, tzinfo=datetime.timezone.utc) 
content="Once upon a time in a cozy little village, there lived a curious cat named Whiskers. With sleek gray fur and bright green eyes, Whiskers loved to explore every nook and cranny of his neighborhood. Every morning, he would walk along the cobblestone streets, chasing butterflies and climbing trees.\n\nOne day, Whiskers discovered a mysterious alleyway he'd never seen before. Filled with excitement, he padded softly into the shadows. At the end of the alley, he found a tiny, abandoned kitten shivering beneath a pile of leaves. Without hesitation, Whiskers gently nudged the little one to comfort him.\n\nFrom that day on, Whiskers and the tiny kitten, whom he named Muffin, became the best of friends. They spent their days playing in the gardens, resting in the sun, and sharing adventures. Whiskers showed Muffin the wonders of their village, and together, they learned that friendship and kindness could brighten even the dullest days.\n\nAnd so, in their little world, Whiskers and Muffin lived happily, reminding everyone that sometimes, the greatest adventures begin with a simple act of kindness." type='TextMessage'

source='B' models_usage=RequestUsage(prompt_tokens=263, completion_tokens=325) metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 49, 25, 5267, tzinfo=datetime.timezone.utc) 
content='从前，在一个温馨的小村庄里，住着一只好奇的猫，名字叫做胡须。胡须拥有光滑的灰色毛发和明亮的绿色眼睛，他喜欢探索家附近的每一个角落、每一条巷子。每天早晨，他都会沿着鹅卵石街道散步，追逐蝴蝶，爬上树枝。\n\n有一天，胡须发现了一条他从未见过的神秘巷子。充满了兴奋，他悄悄地走进阴影中。在巷子的尽头，他发现了一只微微颤抖、被叶子堆包裹着的迷你流浪小猫。胡须毫不犹豫地轻轻用鼻子推动着小猫，就像在安慰他一样。\n\n从那天起，胡须和他给小猫取名叫玛芬，成为了最好的朋友。他们白天在花园里玩耍，阳光下休息，还一起展开冒险。胡须带领玛芬探索村庄的奇迹，教他发现生活中的美好。两个小伙伴在一起的日子，让他们明白了友谊和善良可以让最平凡的日子变得不平凡。\n\n就这样，胡须和玛芬在他们的小世界里快乐地生活着，向所有人证明，有时候，最美好的冒险始于一个善意的简单举动。' type='TextMessage'

source='C' models_usage=RequestUsage(prompt_tokens=593, completion_tokens=236) metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 49, 26, 455880, tzinfo=datetime.timezone.utc) 
content='Once upon a time, in a cozy little village, there lived a curious cat named Whiskers. Whiskers had sleek gray fur and bright green eyes. He loved exploring every corner of his neighborhood—chasing butterflies and climbing trees along the cobblestone streets.\n\nOne day, Whiskers discovered a mysterious alley he had never seen before. Filled with excitement, he quietly crept into the shadows. At the end of the alley, he found a tiny, trembling kitten covered with a pile of leaves. Without hesitation, Whiskers gently nudged the little one to comfort him.\n\nFrom that day on, Whiskers and the kitten, whom he named Muffin, became the best of friends. They spent their days playing in the gardens, resting in the sunshine, and going on adventures together. Whiskers showed Muffin the wonders of their village, teaching him to find beauty in everyday life. Their days together taught them that friendship and kindness can turn even ordinary moments into something special.\n\nAnd so, Whiskers and Muffin happily lived in their little world, proving to everyone that sometimes, the greatest adventures begin with a simple act of kindness.' type='TextMessage'

source='DiGraphStopAgent' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 49, 26, 457883, tzinfo=datetime.timezone.utc) 
content='Digraph execution is complete' type='StopMessage'

messages=[TextMessage(id='e1eb7871-4d60-4025-9989-2d194e9e9820', 
                      
                      source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 49, 21, 301116, tzinfo=datetime.timezone.utc), 
                      content='Write a short story about a cat.', type='TextMessage'), TextMessage(id='9b1f81e1-37ac-4381-8051-acc5d0eae8a3', 
                      
                      source='A', models_usage=RequestUsage(prompt_tokens=26, completion_tokens=233), metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 49, 22, 861842, tzinfo=datetime.timezone.utc), 
                      content="Once upon a time in a cozy little village, there lived a curious cat named Whiskers. With sleek gray fur and bright green eyes, Whiskers loved to explore every nook and cranny of his neighborhood. Every morning, he would walk along the cobblestone streets, chasing butterflies and climbing trees.\n\nOne day, Whiskers discovered a mysterious alleyway he'd never seen before. Filled with excitement, he padded softly into the shadows. At the end of the alley, he found a tiny, abandoned kitten shivering beneath a pile of leaves. Without hesitation, Whiskers gently nudged the little one to comfort him.\n\nFrom that day on, Whiskers and the tiny kitten, whom he named Muffin, became the best of friends. They spent their days playing in the gardens, resting in the sun, and sharing adventures. Whiskers showed Muffin the wonders of their village, and together, they learned that friendship and kindness could brighten even the dullest days.\n\nAnd so, in their little world, Whiskers and Muffin lived happily, reminding everyone that sometimes, the greatest adventures begin with a simple act of kindness.", type='TextMessage'), TextMessage(id='9117d3d2-10c8-4290-a523-ff2301bcdccd', 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    
                      source='B', models_usage=RequestUsage(prompt_tokens=263, completion_tokens=325), metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 49, 25, 5267, tzinfo=datetime.timezone.utc), 
                      content='从前，在一个温馨的小村庄里，住着一只好奇的猫，名字叫做胡须。胡须拥有光滑的灰色毛发和明亮的绿色眼睛，他喜欢探索家附近的每一个角落、每一条巷子。每天早晨，他都会沿着鹅卵石街道散步，追逐蝴蝶，爬上树枝。\n\n有一天，胡须发现了一条他从未见过的神秘巷子。充满了兴奋，他悄悄地走进阴影中。在巷子的尽头，他发现了一只微微颤抖、被叶子堆包裹着的迷你流浪小猫。胡须毫不犹豫地轻轻用鼻子推动着小猫，就像在安慰他一样。\n\n从那天起，胡须和他给小猫取名叫玛芬，成为了最好的朋友。他们白天在花园里玩耍，阳光下休息，还一起展开冒险。胡须带领玛芬探索村庄的奇迹，教他发现生活中的美好。两个小伙伴在一起的日子，让他们明白了友谊和善良可以让最平凡的日子变得不平凡。\n\n就这样，胡须和玛芬在他们的小世界里快乐地生活着，向所有人证明，有时候，最美好的冒险始于一个善意的简单举动。', type='TextMessage'), TextMessage(id='7168a916-7efa-42cd-b5d6-8bddf06dfc95', 
                                                                                                                                                                                                                                                                                                                                                                                                                                                            
                      source='C', models_usage=RequestUsage(prompt_tokens=593, completion_tokens=236), metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 49, 26, 455880, tzinfo=datetime.timezone.utc), 
                      content='Once upon a time, in a cozy little village, there lived a curious cat named Whiskers. Whiskers had sleek gray fur and bright green eyes. He loved exploring every corner of his neighborhood—chasing butterflies and climbing trees along the cobblestone streets.\n\nOne day, Whiskers discovered a mysterious alley he had never seen before. Filled with excitement, he quietly crept into the shadows. At the end of the alley, he found a tiny, trembling kitten covered with a pile of leaves. Without hesitation, Whiskers gently nudged the little one to comfort him.\n\nFrom that day on, Whiskers and the kitten, whom he named Muffin, became the best of friends. They spent their days playing in the gardens, resting in the sunshine, and going on adventures together. Whiskers showed Muffin the wonders of their village, teaching him to find beauty in everyday life. Their days together taught them that friendship and kindness can turn even ordinary moments into something special.\n\nAnd so, Whiskers and Muffin happily lived in their little world, proving to everyone that sometimes, the greatest adventures begin with a simple act of kindness.', type='TextMessage'), 
                      
                      StopMessage(id='341e7521-12d9-4deb-b755-5693aa8ebad4', source='DiGraphStopAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 49, 26, 457883, tzinfo=datetime.timezone.utc), 
                      content='Digraph execution is complete', type='StopMessage')] 

stop_reason='Stop message received, Maximum number of messages 5 reached, current message count: 5'

#### Example 2.

**`Parallel Fan-out: A → (B, C)`**

In [34]:
import asyncio

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow
from autogen_ext.models.openai import OpenAIChatCompletionClient


# async def main():
    # Initialize agents with OpenAI model clients.
model_client = OpenAIChatCompletionClient(model="gpt-4.1-nano")
agent_a = AssistantAgent("A", model_client=model_client, system_message="You are a helpful assistant.")
agent_b = AssistantAgent("B", model_client=model_client, system_message="Translate input to Chinese.")
agent_c = AssistantAgent("C", model_client=model_client, system_message="Translate input to Japanese.")

# Create a directed graph with fan-out flow A -> (B, C).
builder = DiGraphBuilder()
builder.add_node(agent_a).add_node(agent_b).add_node(agent_c)
builder.add_edge(agent_a, agent_b).add_edge(agent_a, agent_c)
graph = builder.build()

# Create a GraphFlow team with the directed graph.
team = GraphFlow(
    participants=[agent_a, agent_b, agent_c],
    graph=graph,
    termination_condition=MaxMessageTermination(5),
)

# Run the team and print the events.
async for event in team.run_stream(task="Write a short story about a cat."):
    print(event)


# asyncio.run(main())

id='a107d437-7c10-4d8c-bf13-cbe7104a0f0c' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 48, 45, 980424, tzinfo=datetime.timezone.utc) content='Write a short story about a cat.' type='TextMessage'
id='fd54cf3d-e30a-49ed-b538-a00463c6ec0c' source='A' models_usage=RequestUsage(prompt_tokens=26, completion_tokens=186) metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 48, 47, 355901, tzinfo=datetime.timezone.utc) content='Once upon a time in a quiet village, there lived a small gray cat named Whiskers. Whiskers was known for her curious nature and bright green eyes that sparkled with mischief. Every morning, she would explore the backyard, chasing butterflies and climbing trees. One sunny afternoon, she discovered a tiny opening behind a bush that led into a mysterious garden filled with colorful flowers and strange little creatures. \n\nIntrigued, Whiskers ventured inside, her nose twitching with excitement. She met a friendly rabbit named 

##### Structured Output - I

**`Parallel Fan-out: A → (B, C)`**

In [ ]:
source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 20, 50, 639650, tzinfo=datetime.timezone.utc) 
content='Write a short story about a cat.' type='TextMessage'

source='A' models_usage=RequestUsage(prompt_tokens=26, completion_tokens=231) metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 20, 53, 558481, tzinfo=datetime.timezone.utc) 
content='Once upon a time, in a cozy little village, there lived a curious cat named Whiskers. With soft gray fur and sparkling green eyes, Whiskers loved to explore every nook and cranny of the neighborhood. Every morning, she would perch on the windowsill, watching the hustle and bustle outside, dreaming of adventures.\n\nOne bright sunny day, Whiskers decided to follow a fluttering butterfly into the nearby woods. She tiptoed past tall trees, over gentle streams, and past cheerful birds singing their melodies. Suddenly, she found herself in a clearing where a tiny, abandoned kitten sat trembling.\n\nWithout hesitation, Whiskers approached and nuzzled the tiny kitten. "Don\'t be afraid," she purred softly. "I\'ll take care of you." From that day on, Whiskers and the little kitten, named Poppy, became inseparable friends. They explored the woods together, shared naps in the sun, and discovered that the greatest treasures were the friends they made along the way.\n\nAnd so, in their little village, the brave and kind-hearted Whiskers lived happily, always ready for the next adventure.' type='TextMessage'

source='B' models_usage=RequestUsage(prompt_tokens=261, completion_tokens=323) metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 20, 56, 643055, tzinfo=datetime.timezone.utc) 
content='从前，在一个温馨的小村庄里，住着一只好奇心旺盛的猫，名字叫胡椒。胡椒拥有柔软的灰色毛发和闪闪发光的绿色眼睛，它喜欢探险，探索每一个角落和缝隙。每天早晨，胡椒都会站在窗台上，看着外面热闹繁忙的景象，梦想着有一天能去冒险。\n\n一天阳光明媚的早晨，胡椒决定跟随一只翩翩起舞的蝴蝶，进入附近的森林。它轻声细步，穿过高高的树木，越过潺潺的小溪，途中还听到愉快的鸟鸣声。突然，在一片空地上，胡椒遇到了一只小猫咪，坐在那里颤抖着，显得很孤单。\n\n胡椒毫不犹豫地走过去，用头轻轻蹭了蹭那只小猫咪。“别害怕，”它轻声说道，“我会照顾你的。”从那天起，胡椒和这只名叫波比的小猫成为了好朋友。它们一起探索森林，一起在阳光下打盹，还渐渐发现，人生中最大的宝藏，就是一路上遇到的朋友。\n\n就这样，勇敢又善良的胡椒在它的小村庄里快乐地生活着，总是准备迎接下一次的冒险。' type='TextMessage'

source='C' models_usage=RequestUsage(prompt_tokens=261, completion_tokens=383) metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 20, 56, 717019, tzinfo=datetime.timezone.utc) 
content='昔々、小さな村に好奇心旺盛な猫のウィスカーズが住んでいました。ふわふわの灰色の毛とキラキラ輝く緑の目を持つウィスカーズは、近所の隅々まで探索するのが大好きでした。毎朝、彼女は窓辺に座り、外の忙しさを眺めながら冒険を夢見ていました。\n\nある晴れた日、ウィスカーズはひらひらと舞う蝶を追いかけて、近くの森へ入りました。背の高い木々をすり抜け、小川を越え、楽しそうにさえずる鳥たちの歌声を聴きながら進みました。突然、彼女は小さく怯える捨てられた子猫がいる空き地にたどり着きました。\n\nためらうことなく、ウィスカーズは近づき、その子猫の顔をなでました。「怖くないよ」と優しく喉を鳴らしながら言いました。「私はあなたの面倒をみるわ。」それ以来、ウィスカーズと小さな子猫のポピーはかけがえのない友達になりました。ふたりは一緒に森を探検し、日向でお昼寝をしながら、最も貴重な宝物は仲間だということを学びました。\n\nそして、小さな村で、勇敢で優しい心を持つウィスカーズは、いつまでも幸せに暮らし、次の冒険をいつでも待ちかまえていました。' type='TextMessage'

source='DiGraphStopAgent' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 20, 56, 718019, tzinfo=datetime.timezone.utc) 
content='Digraph execution is complete' type='StopMessage'

messages=[TextMessage(id='7d23586a-6ec7-4b14-83ba-b6dbc2409c6d', 
                      
                      source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 20, 50, 639650, tzinfo=datetime.timezone.utc), 
                      content='Write a short story about a cat.', type='TextMessage'), TextMessage(id='709761cf-f197-4c2f-8de1-5d4fb6db9c9a', 
                                                                                                   
                      source='A', models_usage=RequestUsage(prompt_tokens=26, completion_tokens=231), metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 20, 53, 558481, tzinfo=datetime.timezone.utc), 
                      content='Once upon a time, in a cozy little village, there lived a curious cat named Whiskers. With soft gray fur and sparkling green eyes, Whiskers loved to explore every nook and cranny of the neighborhood. Every morning, she would perch on the windowsill, watching the hustle and bustle outside, dreaming of adventures.\n\nOne bright sunny day, Whiskers decided to follow a fluttering butterfly into the nearby woods. She tiptoed past tall trees, over gentle streams, and past cheerful birds singing their melodies. Suddenly, she found herself in a clearing where a tiny, abandoned kitten sat trembling.\n\nWithout hesitation, Whiskers approached and nuzzled the tiny kitten. "Don\'t be afraid," she purred softly. "I\'ll take care of you." From that day on, Whiskers and the little kitten, named Poppy, became inseparable friends. They explored the woods together, shared naps in the sun, and discovered that the greatest treasures were the friends they made along the way.\n\nAnd so, in their little village, the brave and kind-hearted Whiskers lived happily, always ready for the next adventure.', type='TextMessage'), TextMessage(id='03cef134-4405-4fa8-8fec-6d14b6400736', 
                      
                      source='B', models_usage=RequestUsage(prompt_tokens=261, completion_tokens=323), metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 20, 56, 643055, tzinfo=datetime.timezone.utc), 
                      content='从前，在一个温馨的小村庄里，住着一只好奇心旺盛的猫，名字叫胡椒。胡椒拥有柔软的灰色毛发和闪闪发光的绿色眼睛，它喜欢探险，探索每一个角落和缝隙。每天早晨，胡椒都会站在窗台上，看着外面热闹繁忙的景象，梦想着有一天能去冒险。\n\n一天阳光明媚的早晨，胡椒决定跟随一只翩翩起舞的蝴蝶，进入附近的森林。它轻声细步，穿过高高的树木，越过潺潺的小溪，途中还听到愉快的鸟鸣声。突然，在一片空地上，胡椒遇到了一只小猫咪，坐在那里颤抖着，显得很孤单。\n\n胡椒毫不犹豫地走过去，用头轻轻蹭了蹭那只小猫咪。“别害怕，”它轻声说道，“我会照顾你的。”从那天起，胡椒和这只名叫波比的小猫成为了好朋友。它们一起探索森林，一起在阳光下打盹，还渐渐发现，人生中最大的宝藏，就是一路上遇到的朋友。\n\n就这样，勇敢又善良的胡椒在它的小村庄里快乐地生活着，总是准备迎接下一次的冒险。', type='TextMessage'), TextMessage(id='fb092779-4d3b-44b8-b1a3-4c2fc8cc0b30', 
                      
                      source='C', models_usage=RequestUsage(prompt_tokens=261, completion_tokens=383), metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 20, 56, 717019, tzinfo=datetime.timezone.utc), 
                      content='昔々、小さな村に好奇心旺盛な猫のウィスカーズが住んでいました。ふわふわの灰色の毛とキラキラ輝く緑の目を持つウィスカーズは、近所の隅々まで探索するのが大好きでした。毎朝、彼女は窓辺に座り、外の忙しさを眺めながら冒険を夢見ていました。\n\nある晴れた日、ウィスカーズはひらひらと舞う蝶を追いかけて、近くの森へ入りました。背の高い木々をすり抜け、小川を越え、楽しそうにさえずる鳥たちの歌声を聴きながら進みました。突然、彼女は小さく怯える捨てられた子猫がいる空き地にたどり着きました。\n\nためらうことなく、ウィスカーズは近づき、その子猫の顔をなでました。「怖くないよ」と優しく喉を鳴らしながら言いました。「私はあなたの面倒をみるわ。」それ以来、ウィスカーズと小さな子猫のポピーはかけがえのない友達になりました。ふたりは一緒に森を探検し、日向でお昼寝をしながら、最も貴重な宝物は仲間だということを学びました。\n\nそして、小さな村で、勇敢で優しい心を持つウィスカーズは、いつまでも幸せに暮らし、次の冒険をいつでも待ちかまえていました。', type='TextMessage'), StopMessage(id='608170ef-9a8c-4d50-9b1b-ad89d166ef47', source='DiGraphStopAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 20, 56, 718019, tzinfo=datetime.timezone.utc), content='Digraph execution is complete', type='StopMessage')] 

stop_reason='Stop message received'

##### Jupyter's Output

id='0c01e941-b269-42c1-9572-26132c1b2d32' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 32, 47, 640199, tzinfo=datetime.timezone.utc) content='Write a short story about a cat.' type='TextMessage'
id='3e207bfa-dc22-44b3-8b69-0e4b55fd30e7' source='A' models_usage=RequestUsage(prompt_tokens=26, completion_tokens=227) metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 32, 49, 953720, tzinfo=datetime.timezone.utc) content='Once upon a time, in a cozy village, there lived a curious cat named Whiskers. With sleek gray fur and bright green eyes, Whiskers loved to explore every nook and cranny of the town. Every morning, he would venture out of his sunny home, chasing butterflies and climbing tall trees.\n\nOne afternoon, Whiskers discovered a hidden alleyway lined with colorful flowers and shimmering stones. Intrigued, he padded quietly inside. To his surprise, he found a tiny door at the base of a cobblestone wall. It was just big enough for him to squeeze through!\n\nOn the other side, Whiskers found a magical garden filled with singing birds and glittering fountains. There, he made friends with a wise old owl and a cheerful rabbit. They shared stories and treats, and Whiskers realized that sometimes, the greatest adventures happen when you follow your curiosity.\n\nFrom then on, Whiskers kept exploring, always eager to find new friends and hidden treasures. And every evening, he returned home grateful for his marvelous, mysterious world — a place where adventure and friendship awaited at every turn.' type='TextMessage'
id='d2dabe67-8c5d-4356-a48d-be109e02b390' source='C' models_usage=RequestUsage(prompt_tokens=257, completion_tokens=371) metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 32, 52, 181335, tzinfo=datetime.timezone.utc) content='昔々、温かい村に、好奇心旺盛な猫のウィスカーズがいました。滑らかな灰色の毛と明るい緑色の目を持つウィスカーズは、町の隅々まで探検するのが大好きでした。毎朝、彼は日当たりの良い家から出かけ、蝶々を追いかけたり、高い木に登ったりして過ごしました。\n\nある日の午後、ウィスカーズは、色とりどりの花や輝く石が並ぶ隠れた路地を見つけました。興味津々で静かに歩みを進めると、石畳の壁の根元に小さな扉を見つけました。それは、しっかりと通れるくらいの大きさでした！\n\n扉の向こうには、歌う鳥たちやきらめく噴水がある魔法の庭が広がっていました。そこでは、賢い老フクロウや陽気なウサギと友達になり、物語やおやつを分かち合いました。ウィスカーズは、時には好奇心を追いかけることで、最高の冒険が生まれることに気づきました。\n\nそれ以来、ウィスカーズは探検を続け、新しい友達や隠された宝物を見つけるのを楽しみました。そして毎晩、彼は素晴らしく神秘的な世界に感謝しながら家に帰りました――冒険と友情がいつも待っている場所です。' type='TextMessage'
id='75566187-205c-4016-a48d-060028c8a5b8' source='B' models_usage=RequestUsage(prompt_tokens=257, completion_tokens=306) metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 32, 52, 614280, tzinfo=datetime.timezone.utc) content='从前，在一个温馨的小村庄里，住着一只好奇心旺盛的猫，名叫胡须。它毛色灰亮，眼睛碧绿，喜欢探索镇上的每一个角落和缝隙。每天早晨，它都会离开温暖的家，追逐蝴蝶，攀爬高高的大树。\n\n一天午后，胡须发现了一条隐藏的小巷，巷子里布满了五彩缤纷的花朵和闪闪发光的石子。它悄悄地走了进去，出乎意料地，在一堵铺满鹅卵石的墙根，发现了一扇微小的门。门刚好够它挤进去！\n\n门那边，是一个神奇的花园，里面有歌唱的鸟儿和闪烁的喷泉。这里，胡须结识了一只智慧的老猫头鹰和一只欢快的兔子。它们分享故事和点心，让胡须明白，有时候，最大的冒险就是跟随自己的好奇心。\n\n从那以后，胡须不断探索，渴望结交新朋友，寻找隐藏的宝藏。每个晚上，它都心怀感激地回到家，庆幸自己拥有一个神奇又迷人的世界——一个冒险与友谊在每个转角等待的地方。' type='TextMessage'
id='bb3f4d4b-b420-4e91-ad01-7cb5559a73a4' source='DiGraphStopAgent' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 32, 52, 616277, tzinfo=datetime.timezone.utc) content='Digraph execution is complete' type='StopMessage'
messages=[TextMessage(id='0c01e941-b269-42c1-9572-26132c1b2d32', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 32, 47, 640199, tzinfo=datetime.timezone.utc), content='Write a short story about a cat.', type='TextMessage'), TextMessage(id='3e207bfa-dc22-44b3-8b69-0e4b55fd30e7', source='A', models_usage=RequestUsage(prompt_tokens=26, completion_tokens=227), metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 32, 49, 953720, tzinfo=datetime.timezone.utc), content='Once upon a time, in a cozy village, there lived a curious cat named Whiskers. With sleek gray fur and bright green eyes, Whiskers loved to explore every nook and cranny of the town. Every morning, he would venture out of his sunny home, chasing butterflies and climbing tall trees.\n\nOne afternoon, Whiskers discovered a hidden alleyway lined with colorful flowers and shimmering stones. Intrigued, he padded quietly inside. To his surprise, he found a tiny door at the base of a cobblestone wall. It was just big enough for him to squeeze through!\n\nOn the other side, Whiskers found a magical garden filled with singing birds and glittering fountains. There, he made friends with a wise old owl and a cheerful rabbit. They shared stories and treats, and Whiskers realized that sometimes, the greatest adventures happen when you follow your curiosity.\n\nFrom then on, Whiskers kept exploring, always eager to find new friends and hidden treasures. And every evening, he returned home grateful for his marvelous, mysterious world — a place where adventure and friendship awaited at every turn.', type='TextMessage'), TextMessage(id='d2dabe67-8c5d-4356-a48d-be109e02b390', source='C', models_usage=RequestUsage(prompt_tokens=257, completion_tokens=371), metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 32, 52, 181335, tzinfo=datetime.timezone.utc), content='昔々、温かい村に、好奇心旺盛な猫のウィスカーズがいました。滑らかな灰色の毛と明るい緑色の目を持つウィスカーズは、町の隅々まで探検するのが大好きでした。毎朝、彼は日当たりの良い家から出かけ、蝶々を追いかけたり、高い木に登ったりして過ごしました。\n\nある日の午後、ウィスカーズは、色とりどりの花や輝く石が並ぶ隠れた路地を見つけました。興味津々で静かに歩みを進めると、石畳の壁の根元に小さな扉を見つけました。それは、しっかりと通れるくらいの大きさでした！\n\n扉の向こうには、歌う鳥たちやきらめく噴水がある魔法の庭が広がっていました。そこでは、賢い老フクロウや陽気なウサギと友達になり、物語やおやつを分かち合いました。ウィスカーズは、時には好奇心を追いかけることで、最高の冒険が生まれることに気づきました。\n\nそれ以来、ウィスカーズは探検を続け、新しい友達や隠された宝物を見つけるのを楽しみました。そして毎晩、彼は素晴らしく神秘的な世界に感謝しながら家に帰りました――冒険と友情がいつも待っている場所です。', type='TextMessage'), TextMessage(id='75566187-205c-4016-a48d-060028c8a5b8', source='B', models_usage=RequestUsage(prompt_tokens=257, completion_tokens=306), metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 32, 52, 614280, tzinfo=datetime.timezone.utc), content='从前，在一个温馨的小村庄里，住着一只好奇心旺盛的猫，名叫胡须。它毛色灰亮，眼睛碧绿，喜欢探索镇上的每一个角落和缝隙。每天早晨，它都会离开温暖的家，追逐蝴蝶，攀爬高高的大树。\n\n一天午后，胡须发现了一条隐藏的小巷，巷子里布满了五彩缤纷的花朵和闪闪发光的石子。它悄悄地走了进去，出乎意料地，在一堵铺满鹅卵石的墙根，发现了一扇微小的门。门刚好够它挤进去！\n\n门那边，是一个神奇的花园，里面有歌唱的鸟儿和闪烁的喷泉。这里，胡须结识了一只智慧的老猫头鹰和一只欢快的兔子。它们分享故事和点心，让胡须明白，有时候，最大的冒险就是跟随自己的好奇心。\n\n从那以后，胡须不断探索，渴望结交新朋友，寻找隐藏的宝藏。每个晚上，它都心怀感激地回到家，庆幸自己拥有一个神奇又迷人的世界——一个冒险与友谊在每个转角等待的地方。', type='TextMessage'), StopMessage(id='bb3f4d4b-b420-4e91-ad01-7cb5559a73a4', source='DiGraphStopAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 32, 52, 616277, tzinfo=datetime.timezone.utc), content='Digraph execution is complete', type='StopMessage')] stop_reason='Stop message received'

##### Structured Output-II

**`Parallel Fan-out: A → (C, B)`**

In [ ]:
source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 32, 47, 640199, tzinfo=datetime.timezone.utc) 
content='Write a short story about a cat.' type='TextMessage'

source='A' models_usage=RequestUsage(prompt_tokens=26, completion_tokens=227) metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 32, 49, 953720, tzinfo=datetime.timezone.utc) 
content='Once upon a time, in a cozy village, there lived a curious cat named Whiskers. With sleek gray fur and bright green eyes, Whiskers loved to explore every nook and cranny of the town. Every morning, he would venture out of his sunny home, chasing butterflies and climbing tall trees.\n\nOne afternoon, Whiskers discovered a hidden alleyway lined with colorful flowers and shimmering stones. Intrigued, he padded quietly inside. To his surprise, he found a tiny door at the base of a cobblestone wall. It was just big enough for him to squeeze through!\n\nOn the other side, Whiskers found a magical garden filled with singing birds and glittering fountains. There, he made friends with a wise old owl and a cheerful rabbit. They shared stories and treats, and Whiskers realized that sometimes, the greatest adventures happen when you follow your curiosity.\n\nFrom then on, Whiskers kept exploring, always eager to find new friends and hidden treasures. And every evening, he returned home grateful for his marvelous, mysterious world — a place where adventure and friendship awaited at every turn.' type='TextMessage'

source='C' models_usage=RequestUsage(prompt_tokens=257, completion_tokens=371) metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 32, 52, 181335, tzinfo=datetime.timezone.utc) 
content='昔々、温かい村に、好奇心旺盛な猫のウィスカーズがいました。滑らかな灰色の毛と明るい緑色の目を持つウィスカーズは、町の隅々まで探検するのが大好きでした。毎朝、彼は日当たりの良い家から出かけ、蝶々を追いかけたり、高い木に登ったりして過ごしました。\n\nある日の午後、ウィスカーズは、色とりどりの花や輝く石が並ぶ隠れた路地を見つけました。興味津々で静かに歩みを進めると、石畳の壁の根元に小さな扉を見つけました。それは、しっかりと通れるくらいの大きさでした！\n\n扉の向こうには、歌う鳥たちやきらめく噴水がある魔法の庭が広がっていました。そこでは、賢い老フクロウや陽気なウサギと友達になり、物語やおやつを分かち合いました。ウィスカーズは、時には好奇心を追いかけることで、最高の冒険が生まれることに気づきました。\n\nそれ以来、ウィスカーズは探検を続け、新しい友達や隠された宝物を見つけるのを楽しみました。そして毎晩、彼は素晴らしく神秘的な世界に感謝しながら家に帰りました――冒険と友情がいつも待っている場所です。' type='TextMessage'

source='B' models_usage=RequestUsage(prompt_tokens=257, completion_tokens=306) metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 32, 52, 614280, tzinfo=datetime.timezone.utc) 
content='从前，在一个温馨的小村庄里，住着一只好奇心旺盛的猫，名叫胡须。它毛色灰亮，眼睛碧绿，喜欢探索镇上的每一个角落和缝隙。每天早晨，它都会离开温暖的家，追逐蝴蝶，攀爬高高的大树。\n\n一天午后，胡须发现了一条隐藏的小巷，巷子里布满了五彩缤纷的花朵和闪闪发光的石子。它悄悄地走了进去，出乎意料地，在一堵铺满鹅卵石的墙根，发现了一扇微小的门。门刚好够它挤进去！\n\n门那边，是一个神奇的花园，里面有歌唱的鸟儿和闪烁的喷泉。这里，胡须结识了一只智慧的老猫头鹰和一只欢快的兔子。它们分享故事和点心，让胡须明白，有时候，最大的冒险就是跟随自己的好奇心。\n\n从那以后，胡须不断探索，渴望结交新朋友，寻找隐藏的宝藏。每个晚上，它都心怀感激地回到家，庆幸自己拥有一个神奇又迷人的世界——一个冒险与友谊在每个转角等待的地方。' type='TextMessage'

source='DiGraphStopAgent' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 32, 52, 616277, tzinfo=datetime.timezone.utc) 
content='Digraph execution is complete' type='StopMessage'

messages=[TextMessage(id='0c01e941-b269-42c1-9572-26132c1b2d32',
                       
                      source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 32, 47, 640199, tzinfo=datetime.timezone.utc), 
                      content='Write a short story about a cat.', type='TextMessage'), TextMessage(id='3e207bfa-dc22-44b3-8b69-0e4b55fd30e7', 
                      
                      source='A', models_usage=RequestUsage(prompt_tokens=26, completion_tokens=227), metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 32, 49, 953720, tzinfo=datetime.timezone.utc), 
                      content='Once upon a time, in a cozy village, there lived a curious cat named Whiskers. With sleek gray fur and bright green eyes, Whiskers loved to explore every nook and cranny of the town. Every morning, he would venture out of his sunny home, chasing butterflies and climbing tall trees.\n\nOne afternoon, Whiskers discovered a hidden alleyway lined with colorful flowers and shimmering stones. Intrigued, he padded quietly inside. To his surprise, he found a tiny door at the base of a cobblestone wall. It was just big enough for him to squeeze through!\n\nOn the other side, Whiskers found a magical garden filled with singing birds and glittering fountains. There, he made friends with a wise old owl and a cheerful rabbit. They shared stories and treats, and Whiskers realized that sometimes, the greatest adventures happen when you follow your curiosity.\n\nFrom then on, Whiskers kept exploring, always eager to find new friends and hidden treasures. And every evening, he returned home grateful for his marvelous, mysterious world — a place where adventure and friendship awaited at every turn.', type='TextMessage'), TextMessage(id='d2dabe67-8c5d-4356-a48d-be109e02b390', 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            
                      source='C', models_usage=RequestUsage(prompt_tokens=257, completion_tokens=371), metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 32, 52, 181335, tzinfo=datetime.timezone.utc), 
                      content='昔々、温かい村に、好奇心旺盛な猫のウィスカーズがいました。滑らかな灰色の毛と明るい緑色の目を持つウィスカーズは、町の隅々まで探検するのが大好きでした。毎朝、彼は日当たりの良い家から出かけ、蝶々を追いかけたり、高い木に登ったりして過ごしました。\n\nある日の午後、ウィスカーズは、色とりどりの花や輝く石が並ぶ隠れた路地を見つけました。興味津々で静かに歩みを進めると、石畳の壁の根元に小さな扉を見つけました。それは、しっかりと通れるくらいの大きさでした！\n\n扉の向こうには、歌う鳥たちやきらめく噴水がある魔法の庭が広がっていました。そこでは、賢い老フクロウや陽気なウサギと友達になり、物語やおやつを分かち合いました。ウィスカーズは、時には好奇心を追いかけることで、最高の冒険が生まれることに気づきました。\n\nそれ以来、ウィスカーズは探検を続け、新しい友達や隠された宝物を見つけるのを楽しみました。そして毎晩、彼は素晴らしく神秘的な世界に感謝しながら家に帰りました――冒険と友情がいつも待っている場所です。', type='TextMessage'), TextMessage(id='75566187-205c-4016-a48d-060028c8a5b8', 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          
                      source='B', models_usage=RequestUsage(prompt_tokens=257, completion_tokens=306), metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 32, 52, 614280, tzinfo=datetime.timezone.utc), 
                      content='从前，在一个温馨的小村庄里，住着一只好奇心旺盛的猫，名叫胡须。它毛色灰亮，眼睛碧绿，喜欢探索镇上的每一个角落和缝隙。每天早晨，它都会离开温暖的家，追逐蝴蝶，攀爬高高的大树。\n\n一天午后，胡须发现了一条隐藏的小巷，巷子里布满了五彩缤纷的花朵和闪闪发光的石子。它悄悄地走了进去，出乎意料地，在一堵铺满鹅卵石的墙根，发现了一扇微小的门。门刚好够它挤进去！\n\n门那边，是一个神奇的花园，里面有歌唱的鸟儿和闪烁的喷泉。这里，胡须结识了一只智慧的老猫头鹰和一只欢快的兔子。它们分享故事和点心，让胡须明白，有时候，最大的冒险就是跟随自己的好奇心。\n\n从那以后，胡须不断探索，渴望结交新朋友，寻找隐藏的宝藏。每个晚上，它都心怀感激地回到家，庆幸自己拥有一个神奇又迷人的世界——一个冒险与友谊在每个转角等待的地方。', type='TextMessage'), StopMessage(id='bb3f4d4b-b420-4e91-ad01-7cb5559a73a4', 
                                                                                                                                                                                                                                                                                                                                                                                                                                     
                      source='DiGraphStopAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 32, 52, 616277, tzinfo=datetime.timezone.utc), 
                      content='Digraph execution is complete', type='StopMessage')] 
stop_reason='Stop message received'

#### Example 3.

**`Sequential Flow: A → (B, C) → D`**

In [ ]:
import asyncio

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow
from autogen_ext.models.openai import OpenAIChatCompletionClient



# Initialize agents with OpenAI model clients.
model_client = OpenAIChatCompletionClient(model="gpt-4.1-nano")
agent_a = AssistantAgent("A", model_client=model_client, system_message="You are a helpful assistant.")
agent_b = AssistantAgent("B", model_client=model_client, system_message="Translate input to Chinese.")
agent_c = AssistantAgent("C", model_client=model_client, system_message="Translate input to Japanese.")
agent_d = AssistantAgent("D", model_client=model_client, system_message="Summarize.")


# Create a directed graph with fan-out flow A -> (B, C) -> D.

builder = DiGraphBuilder()
builder.add_node(agent_a).add_node(agent_b).add_node(agent_c).add_node(agent_d)
builder.add_edge(agent_a, agent_b).add_edge(agent_a, agent_c)
builder.add_edge(agent_b, agent_d).add_edge(agent_c, agent_d)
graph = builder.build()

# Create a GraphFlow team with the directed graph.
team = GraphFlow(
    participants=[agent_a, agent_b, agent_c, agent_d],
    graph=graph,
    termination_condition=MaxMessageTermination(5),
)

# Run the team and print the events.
async for event in team.run_stream(task="Write a short story about a cat."):
    print(event)

id='b43e70ea-89f0-48ea-a4ac-a4b6b4da02f0' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 14, 8, 163406, tzinfo=datetime.timezone.utc) content='Write a short story about a cat.' type='TextMessage'
id='81972e48-16ac-46f9-a6e5-d7c321e7cae7' source='A' models_usage=RequestUsage(prompt_tokens=26, completion_tokens=250) metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 14, 9, 887291, tzinfo=datetime.timezone.utc) content='Once upon a time, in a cozy little village, there was a curious cat named Whiskers. With sleek gray fur and bright green eyes, Whiskers loved to explore every nook and cranny of the neighborhood. Every morning, he set out on adventures, climbing trees, chasing butterflies, and sniffing out new scents.\n\nOne sunny afternoon, Whiskers wandered into a garden he had never seen before. In the middle of the garden stood a small, shimmering pond. As he gazed into the water, he saw his reflection and a world full of shimmering rippl

##### Structured Output-I

**`A → (B, C) → D`**

In [ ]:
source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 51, 49, 377347, tzinfo=datetime.timezone.utc) 
content='Write a short story about a cat.' type='TextMessage'

source='A' models_usage=RequestUsage(prompt_tokens=26, completion_tokens=211) metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 51, 51, 182222, tzinfo=datetime.timezone.utc) 
content='Once upon a time, in a cozy little village, there was a curious cat named Whiskers. With soft grey fur and bright emerald eyes, he loved to explore every nook and cranny of his neighborhood. \n\nOne sunny afternoon, Whiskers discovered a tiny door hidden behind a thick hedge. Intrigued, he nudged it open with his nose and slipped inside. To his surprise, he found himself in a magical garden filled with glowing flowers and singing birds. \n\nIn the center of the garden stood a fountain bubbling with sparkling water. As Whiskers approached, a gentle voice whispered, "Welcome, traveler. You have found the secret garden of dreams." Whiskers purred happily and spent the day chasing shimmering butterflies and napping atop soft moss.\n\nWhen the sun set, Whiskers made his way back home, his heart full of wonder. From that day on, he visited the magical garden whenever he needed a little bit of enchantment, always returning with a joyful purr and a twinkle in his eye.' type='TextMessage'

source='B' models_usage=RequestUsage(prompt_tokens=241, completion_tokens=271) metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 51, 53, 345895, tzinfo=datetime.timezone.utc) 
content='从前，在一个温馨的小村庄里，有一只好奇的猫，名叫胡须。它拥有柔软的灰色毛发和明亮的翡翠色眼睛，喜欢探索周围的每一个角落。\n\n一个阳光明媚的下午，胡须在浓密的树篱后发现了一扇小门。出于好奇，它用鼻子轻轻推开门，溜了进去。出乎意料的是，它发现自己进入了一个神奇的花园，那里开满了发光的花朵，鸟儿在歌唱。\n\n花园的中心有一个喷泉，喷涌出闪烁的水珠。胡须走近时，一个温柔的声音低语道：“欢迎，旅行者。你找到的是梦想的秘密花园。”胡须幸福地呼噜呼噜叫着，整天追逐闪烁的蝴蝶，在柔软的青苔上打盹。\n\n夕阳西下时，胡须满怀惊喜地回到了家中。 从那天起，每当它需要一点魔力时，就会造访这个神奇的花园，总是带着满腹的喜悦和眼中的光芒回家。' type='TextMessage'

source='C' models_usage=RequestUsage(prompt_tokens=241, completion_tokens=353) metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 51, 54, 49032, tzinfo=datetime.timezone.utc) 
content='昔々、小さな村に好奇心旺盛な猫のウィスカーズがいました。ふわふわの灰色の毛と輝くエメラルドの目を持ち、彼は周りのすみずみまで探検するのが大好きでした。\n\nある晴れた午後、ウィスカーズは厚い生垣の behind に隠された小さな扉を見つけました。好奇心に駆られ、鼻でそっと扉を押し開けると、中に入りました。驚いたことに、彼は光る花と歌う鳥たちでいっぱいの魔法の庭園に迷い込みました。\n\n庭の中心には、きらきらとした水を湧き出す噴水が立っていました。ウィスカーズが近づくと、優しい声がささやきました。「ようこそ、旅人さん。あなたは夢の秘密の庭を見つけましたよ。」ウィスカーズは幸せそうに喉を鳴らしながら、その日一日、輝く蝶々を追いかけたり、柔らかな苔の上で昼寝をしたりして過ごしました。\n\n日が暮れると、ウィスカーズは心を満たしながら家に帰りました。その日から、少しの魔法が欲しいときにはいつでも魔法の庭を訪れ、喜びいっぱいに喉を鳴らし、目に輝きを宿して帰ってきました。' type='TextMessage'

source='D' models_usage=RequestUsage(prompt_tokens=874, completion_tokens=225) metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 51, 56, 462040, tzinfo=datetime.timezone.utc) 
content='昔々、小さな村に好奇心いっぱいの猫、ウィスカーズがいました。ふわふわの灰色の毛と輝くエメラルドの目を持ち、探索が大好きでした。ある晴れた日、厚い生垣の後ろに隠された小さな扉を見つけたウィスカーズは、鼻でそっと押し開けて中に入りました。そこは光る花と歌う鳥たちに満ちた魔法の庭でした。庭の中央にはきらめく噴水があり、優しい声が「夢の秘密の庭へようこそ」とささやきました。ウィスカーズは蝶を追いかけたり、苔の上で昼寝をしたりして一日を過ごし、日没とともに家に帰りました。それ以来、魔法の庭は、彼が魔法を求めるときの特別な場所となりました。' type='TextMessage'

source='DiGraphStopAgent' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 8, 51, 56, 464040, tzinfo=datetime.timezone.utc) 
content='Digraph execution is complete' type='StopMessage'

messages=[TextMessage(id='fea5a5cc-1105-4e1f-96f1-cacc2f72483b', 
                      
                      source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 51, 49, 377347, tzinfo=datetime.timezone.utc), 
                      content='Write a short story about a cat.', type='TextMessage'), TextMessage(id='c0aeaa91-6ccf-43c3-bee1-844cfb461a15', 
                                                                                                   
                      source='A', models_usage=RequestUsage(prompt_tokens=26, completion_tokens=211), metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 51, 51, 182222, tzinfo=datetime.timezone.utc), 
                      content='Once upon a time, in a cozy little village, there was a curious cat named Whiskers. With soft grey fur and bright emerald eyes, he loved to explore every nook and cranny of his neighborhood. \n\nOne sunny afternoon, Whiskers discovered a tiny door hidden behind a thick hedge. Intrigued, he nudged it open with his nose and slipped inside. To his surprise, he found himself in a magical garden filled with glowing flowers and singing birds. \n\nIn the center of the garden stood a fountain bubbling with sparkling water. As Whiskers approached, a gentle voice whispered, "Welcome, traveler. You have found the secret garden of dreams." Whiskers purred happily and spent the day chasing shimmering butterflies and napping atop soft moss.\n\nWhen the sun set, Whiskers made his way back home, his heart full of wonder. From that day on, he visited the magical garden whenever he needed a little bit of enchantment, always returning with a joyful purr and a twinkle in his eye.', type='TextMessage'), TextMessage(id='96601ccc-83bc-4ae0-8b78-ad983c89911a', 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 
                      source='B', models_usage=RequestUsage(prompt_tokens=241, completion_tokens=271), metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 51, 53, 345895, tzinfo=datetime.timezone.utc), 
                      content='从前，在一个温馨的小村庄里，有一只好奇的猫，名叫胡须。它拥有柔软的灰色毛发和明亮的翡翠色眼睛，喜欢探索周围的每一个角落。\n\n一个阳光明媚的下午，胡须在浓密的树篱后发现了一扇小门。出于好奇，它用鼻子轻轻推开门，溜了进去。出乎意料的是，它发现自己进入了一个神奇的花园，那里开满了发光的花朵，鸟儿在歌唱。\n\n花园的中心有一个喷泉，喷涌出闪烁的水珠。胡须走近时，一个温柔的声音低语道：“欢迎，旅行者。你找到的是梦想的秘密花园。”胡须幸福地呼噜呼噜叫着，整天追逐闪烁的蝴蝶，在柔软的青苔上打盹。\n\n夕阳西下时，胡须满怀惊喜地回到了家中。 从那天起，每当它需要一点魔力时，就会造访这个神奇的花园，总是带着满腹的喜悦和眼中的光芒回家。', type='TextMessage'), TextMessage(id='699cdac1-50d1-44d3-9d0a-53e357c2cf11', 
                                                                                                                                                                                                                                                                                                                                                                                               
                      source='C', models_usage=RequestUsage(prompt_tokens=241, completion_tokens=353), metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 51, 54, 49032, tzinfo=datetime.timezone.utc), content='昔々、小さな村に好奇心旺盛な猫のウィスカーズがいました。ふわふわの灰色の毛と輝くエメラルドの目を持ち、彼は周りのすみずみまで探検するのが大好きでした。\n\nある晴れた午後、ウィスカーズは厚い生垣の behind に隠された小さな扉を見つけました。好奇心に駆られ、鼻でそっと扉を押し開けると、中に入りました。驚いたことに、彼は光る花と歌う鳥たちでいっぱいの魔法の庭園に迷い込みました。\n\n庭の中心には、きらきらとした水を湧き出す噴水が立っていました。ウィスカーズが近づくと、優しい声がささやきました。「ようこそ、旅人さん。あなたは夢の秘密の庭を見つけましたよ。」ウィスカーズは幸せそうに喉を鳴らしながら、その日一日、輝く蝶々を追いかけたり、柔らかな苔の上で昼寝をしたりして過ごしました。\n\n日が暮れると、ウィスカーズは心を満たしながら家に帰りました。その日から、少しの魔法が欲しいときにはいつでも魔法の庭を訪れ、喜びいっぱいに喉を鳴らし、目に輝きを宿して帰ってきました。', type='TextMessage'), TextMessage(id='1af41bf2-7e4d-4086-bdbc-ac213a0cb0d1', 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   
                      source='D', models_usage=RequestUsage(prompt_tokens=874, completion_tokens=225), metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 51, 56, 462040, tzinfo=datetime.timezone.utc), 
                      content='昔々、小さな村に好奇心いっぱいの猫、ウィスカーズがいました。ふわふわの灰色の毛と輝くエメラルドの目を持ち、探索が大好きでした。ある晴れた日、厚い生垣の後ろに隠された小さな扉を見つけたウィスカーズは、鼻でそっと押し開けて中に入りました。そこは光る花と歌う鳥たちに満ちた魔法の庭でした。庭の中央にはきらめく噴水があり、優しい声が「夢の秘密の庭へようこそ」とささやきました。ウィスカーズは蝶を追いかけたり、苔の上で昼寝をしたりして一日を過ごし、日没とともに家に帰りました。それ以来、魔法の庭は、彼が魔法を求めるときの特別な場所となりました。', type='TextMessage'), StopMessage(id='56b121d6-7603-4dbb-9b0e-1d5f0ec28cfc', 
                                                                                                                                                                                                                                                                                                                                              
                      source='DiGraphStopAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 8, 51, 56, 464040, tzinfo=datetime.timezone.utc), 
                      content='Digraph execution is complete', type='StopMessage')] 

stop_reason='Stop message received, Maximum number of messages 5 reached, current message count: 5'

##### Jupyter's Output

id='6f3b1547-9a42-4bfc-b541-1ff358575253' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 2, 34, 92868, tzinfo=datetime.timezone.utc) content='Write a short story about a cat.' type='TextMessage'
id='226cf3e0-7c9f-47a8-a1c0-d0f4f199a871' source='A' models_usage=RequestUsage(prompt_tokens=26, completion_tokens=210) metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 2, 36, 42370, tzinfo=datetime.timezone.utc) content='Once upon a time, in a cozy little village, there was a curious gray cat named Whiskers. Every morning, Whiskers would wander outside, eager to explore the world beyond his home. One sunny day, he discovered a hidden trail in the woods that he had never seen before.\n\nIntrigued, Whiskers padded along the path, discovering vibrant flowers and chirping birds. Suddenly, he heard a soft rustling in the bushes. Out popped a tiny, shivering squirrel looking for food.\n\nFeeling compassionate, Whiskers gently nudged the squirrel with his nose. Seeing his kind gesture, the squirrel smiled and offered Whiskers a shiny acorn as a token of thanks. From that day on, Whiskers and the squirrel became the best of friends, exploring the woods and sharing adventures together.\n\nAnd so, Whiskers learned that the best treasures in life are friendship and kindness. From that day forward, he continued to explore with an open heart, always eager for new friends and new stories to tell.' type='TextMessage'
id='9ad33779-5256-45e2-ae36-32e9b6d29e99' source='C' models_usage=RequestUsage(prompt_tokens=240, completion_tokens=363) metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 2, 37, 989683, tzinfo=datetime.timezone.utc) content='昔々、居心地の良い小さな村に、好奇心旺盛な灰色の猫、ウィスカーズが住んでいました。毎朝、ウィスカーズは家の外に出て、広がる世界を探検したくてたまりませんでした。ある晴れた日、彼は森の中で今まで見たことのない隠れた小道を発見しました。\n\n興味をそそられたウィスカーズは、その道をゆっくりと歩き始め、色とりどりの花やさえずる鳥たちを見つけました。突然、茂みの中で小さな音がしました。出てきたのは、震える小さなリスで、お腹を空かせている様子でした。\n\n思いやりを感じたウィスカーズは、そっと鼻でリスを撫でました。その優しい行動に感動したリスは笑顔を見せ、感謝の印として光り輝くどんぐりをウィスカーズにプレゼントしました。その日から、ウィスカーズとリスは親友になり、一緒に森を探検し、冒険を分かち合いました。\n\nこうして、ウィスカーズは、人生で最も価値のある宝物は友情と優しさだと学びました。それ以降も、彼は心を開いて探検を続け、新しい友達や素敵な物語を見つけることにいつまでもわくわくしていました。' type='TextMessage'
id='bc728766-2df0-4b83-887b-f9cadb5e8d8d' source='B' models_usage=RequestUsage(prompt_tokens=240, completion_tokens=276) metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 2, 38, 266113, tzinfo=datetime.timezone.utc) content='很久以前，在一个温馨的小村庄里，住着一只好奇的灰色猫，名字叫胡须。每天早晨，胡须都会走出家门，渴望探索外面的世界。有一天晴朗的早晨，他在树林里发现了一条隐藏的小路，这是他以前从未见过的。\n\n出于好奇，胡须轻轻地沿着小路走着，发现了鲜艳的花朵和欢快的鸟鸣。突然，他听到灌木丛里传来轻轻的沙沙声。只见一只瘦小颤抖的松鼠从灌木中探出头，正在寻找食物。\n\n胡须看到它，心生怜悯，温柔地用鼻子轻轻推了推松鼠。松鼠看到胡须的善意微笑了，并赠送给他一颗闪亮的橡子作为感谢。从那天起，胡须和松鼠成了最好的朋友，一起探索森林，分享各种冒险。\n\n从此，胡须明白了，生命中最宝贵的财富是友谊和善良。他带着一颗开放的心继续探索，期待结识更多的朋友，讲述更多的故事。' type='TextMessage'
id='1b3e5db3-bb0f-455c-8793-59f21e9bc8ab' source='D' models_usage=RequestUsage(prompt_tokens=888, completion_tokens=143) metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 2, 39, 704547, tzinfo=datetime.timezone.utc) content='很久以前，在一个温馨的小村庄里，住着好奇的灰猫胡须。每天早晨，他都喜欢探索外面的世界。有一天，他在树林里发现了一条隐藏的小路，沿着小路，他遇见了一只瘦小颤抖的松鼠。胡须用善意的行为帮助了松鼠，作为感谢，松鼠送给他一颗闪亮的橡子。从那以后，他们成为了最好的朋友，一起探索森林，分享冒险。胡须明白，友谊和善良是生命中最宝贵的财富，并继续带着开放的心探索新朋友和故事。' type='TextMessage'
id='9b7ac7bc-c22a-4928-9aea-23599ae31ba6' source='DiGraphStopAgent' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 2, 39, 706552, tzinfo=datetime.timezone.utc) content='Digraph execution is complete' type='StopMessage'
messages=[TextMessage(id='6f3b1547-9a42-4bfc-b541-1ff358575253', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 2, 34, 92868, tzinfo=datetime.timezone.utc), content='Write a short story about a cat.', type='TextMessage'), TextMessage(id='226cf3e0-7c9f-47a8-a1c0-d0f4f199a871', source='A', models_usage=RequestUsage(prompt_tokens=26, completion_tokens=210), metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 2, 36, 42370, tzinfo=datetime.timezone.utc), content='Once upon a time, in a cozy little village, there was a curious gray cat named Whiskers. Every morning, Whiskers would wander outside, eager to explore the world beyond his home. One sunny day, he discovered a hidden trail in the woods that he had never seen before.\n\nIntrigued, Whiskers padded along the path, discovering vibrant flowers and chirping birds. Suddenly, he heard a soft rustling in the bushes. Out popped a tiny, shivering squirrel looking for food.\n\nFeeling compassionate, Whiskers gently nudged the squirrel with his nose. Seeing his kind gesture, the squirrel smiled and offered Whiskers a shiny acorn as a token of thanks. From that day on, Whiskers and the squirrel became the best of friends, exploring the woods and sharing adventures together.\n\nAnd so, Whiskers learned that the best treasures in life are friendship and kindness. From that day forward, he continued to explore with an open heart, always eager for new friends and new stories to tell.', type='TextMessage'), TextMessage(id='9ad33779-5256-45e2-ae36-32e9b6d29e99', source='C', models_usage=RequestUsage(prompt_tokens=240, completion_tokens=363), metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 2, 37, 989683, tzinfo=datetime.timezone.utc), content='昔々、居心地の良い小さな村に、好奇心旺盛な灰色の猫、ウィスカーズが住んでいました。毎朝、ウィスカーズは家の外に出て、広がる世界を探検したくてたまりませんでした。ある晴れた日、彼は森の中で今まで見たことのない隠れた小道を発見しました。\n\n興味をそそられたウィスカーズは、その道をゆっくりと歩き始め、色とりどりの花やさえずる鳥たちを見つけました。突然、茂みの中で小さな音がしました。出てきたのは、震える小さなリスで、お腹を空かせている様子でした。\n\n思いやりを感じたウィスカーズは、そっと鼻でリスを撫でました。その優しい行動に感動したリスは笑顔を見せ、感謝の印として光り輝くどんぐりをウィスカーズにプレゼントしました。その日から、ウィスカーズとリスは親友になり、一緒に森を探検し、冒険を分かち合いました。\n\nこうして、ウィスカーズは、人生で最も価値のある宝物は友情と優しさだと学びました。それ以降も、彼は心を開いて探検を続け、新しい友達や素敵な物語を見つけることにいつまでもわくわくしていました。', type='TextMessage'), TextMessage(id='bc728766-2df0-4b83-887b-f9cadb5e8d8d', source='B', models_usage=RequestUsage(prompt_tokens=240, completion_tokens=276), metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 2, 38, 266113, tzinfo=datetime.timezone.utc), content='很久以前，在一个温馨的小村庄里，住着一只好奇的灰色猫，名字叫胡须。每天早晨，胡须都会走出家门，渴望探索外面的世界。有一天晴朗的早晨，他在树林里发现了一条隐藏的小路，这是他以前从未见过的。\n\n出于好奇，胡须轻轻地沿着小路走着，发现了鲜艳的花朵和欢快的鸟鸣。突然，他听到灌木丛里传来轻轻的沙沙声。只见一只瘦小颤抖的松鼠从灌木中探出头，正在寻找食物。\n\n胡须看到它，心生怜悯，温柔地用鼻子轻轻推了推松鼠。松鼠看到胡须的善意微笑了，并赠送给他一颗闪亮的橡子作为感谢。从那天起，胡须和松鼠成了最好的朋友，一起探索森林，分享各种冒险。\n\n从此，胡须明白了，生命中最宝贵的财富是友谊和善良。他带着一颗开放的心继续探索，期待结识更多的朋友，讲述更多的故事。', type='TextMessage'), TextMessage(id='1b3e5db3-bb0f-455c-8793-59f21e9bc8ab', source='D', models_usage=RequestUsage(prompt_tokens=888, completion_tokens=143), metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 2, 39, 704547, tzinfo=datetime.timezone.utc), content='很久以前，在一个温馨的小村庄里，住着好奇的灰猫胡须。每天早晨，他都喜欢探索外面的世界。有一天，他在树林里发现了一条隐藏的小路，沿着小路，他遇见了一只瘦小颤抖的松鼠。胡须用善意的行为帮助了松鼠，作为感谢，松鼠送给他一颗闪亮的橡子。从那以后，他们成为了最好的朋友，一起探索森林，分享冒险。胡须明白，友谊和善良是生命中最宝贵的财富，并继续带着开放的心探索新朋友和故事。', type='TextMessage'), StopMessage(id='9b7ac7bc-c22a-4928-9aea-23599ae31ba6', source='DiGraphStopAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 2, 39, 706552, tzinfo=datetime.timezone.utc), content='Digraph execution is complete', type='StopMessage')] stop_reason='Stop message received, Maximum number of messages 5 reached, current message count: 5'

##### Structured Output-II

**`A → (C, B) → D`**

In [ ]:
source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 2, 34, 92868, tzinfo=datetime.timezone.utc) 
content='Write a short story about a cat.' type='TextMessage'

source='A' models_usage=RequestUsage(prompt_tokens=26, completion_tokens=210) metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 2, 36, 42370, tzinfo=datetime.timezone.utc) 
content='Once upon a time, in a cozy little village, there was a curious gray cat named Whiskers. Every morning, Whiskers would wander outside, eager to explore the world beyond his home. One sunny day, he discovered a hidden trail in the woods that he had never seen before.\n\nIntrigued, Whiskers padded along the path, discovering vibrant flowers and chirping birds. Suddenly, he heard a soft rustling in the bushes. Out popped a tiny, shivering squirrel looking for food.\n\nFeeling compassionate, Whiskers gently nudged the squirrel with his nose. Seeing his kind gesture, the squirrel smiled and offered Whiskers a shiny acorn as a token of thanks. From that day on, Whiskers and the squirrel became the best of friends, exploring the woods and sharing adventures together.\n\nAnd so, Whiskers learned that the best treasures in life are friendship and kindness. From that day forward, he continued to explore with an open heart, always eager for new friends and new stories to tell.' type='TextMessage'

source='C' models_usage=RequestUsage(prompt_tokens=240, completion_tokens=363) metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 2, 37, 989683, tzinfo=datetime.timezone.utc) 
content='昔々、居心地の良い小さな村に、好奇心旺盛な灰色の猫、ウィスカーズが住んでいました。毎朝、ウィスカーズは家の外に出て、広がる世界を探検したくてたまりませんでした。ある晴れた日、彼は森の中で今まで見たことのない隠れた小道を発見しました。\n\n興味をそそられたウィスカーズは、その道をゆっくりと歩き始め、色とりどりの花やさえずる鳥たちを見つけました。突然、茂みの中で小さな音がしました。出てきたのは、震える小さなリスで、お腹を空かせている様子でした。\n\n思いやりを感じたウィスカーズは、そっと鼻でリスを撫でました。その優しい行動に感動したリスは笑顔を見せ、感謝の印として光り輝くどんぐりをウィスカーズにプレゼントしました。その日から、ウィスカーズとリスは親友になり、一緒に森を探検し、冒険を分かち合いました。\n\nこうして、ウィスカーズは、人生で最も価値のある宝物は友情と優しさだと学びました。それ以降も、彼は心を開いて探検を続け、新しい友達や素敵な物語を見つけることにいつまでもわくわくしていました。' type='TextMessage'

source='B' models_usage=RequestUsage(prompt_tokens=240, completion_tokens=276) metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 2, 38, 266113, tzinfo=datetime.timezone.utc) 
content='很久以前，在一个温馨的小村庄里，住着一只好奇的灰色猫，名字叫胡须。每天早晨，胡须都会走出家门，渴望探索外面的世界。有一天晴朗的早晨，他在树林里发现了一条隐藏的小路，这是他以前从未见过的。\n\n出于好奇，胡须轻轻地沿着小路走着，发现了鲜艳的花朵和欢快的鸟鸣。突然，他听到灌木丛里传来轻轻的沙沙声。只见一只瘦小颤抖的松鼠从灌木中探出头，正在寻找食物。\n\n胡须看到它，心生怜悯，温柔地用鼻子轻轻推了推松鼠。松鼠看到胡须的善意微笑了，并赠送给他一颗闪亮的橡子作为感谢。从那天起，胡须和松鼠成了最好的朋友，一起探索森林，分享各种冒险。\n\n从此，胡须明白了，生命中最宝贵的财富是友谊和善良。他带着一颗开放的心继续探索，期待结识更多的朋友，讲述更多的故事。' type='TextMessage'

source='D' models_usage=RequestUsage(prompt_tokens=888, completion_tokens=143) metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 2, 39, 704547, tzinfo=datetime.timezone.utc) 
content='很久以前，在一个温馨的小村庄里，住着好奇的灰猫胡须。每天早晨，他都喜欢探索外面的世界。有一天，他在树林里发现了一条隐藏的小路，沿着小路，他遇见了一只瘦小颤抖的松鼠。胡须用善意的行为帮助了松鼠，作为感谢，松鼠送给他一颗闪亮的橡子。从那以后，他们成为了最好的朋友，一起探索森林，分享冒险。胡须明白，友谊和善良是生命中最宝贵的财富，并继续带着开放的心探索新朋友和故事。' type='TextMessage'

source='DiGraphStopAgent' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 9, 2, 39, 706552, tzinfo=datetime.timezone.utc) 
content='Digraph execution is complete' type='StopMessage'

messages=[TextMessage(id='6f3b1547-9a42-4bfc-b541-1ff358575253', 
                      
                      source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 2, 34, 92868, tzinfo=datetime.timezone.utc), 
                      content='Write a short story about a cat.', type='TextMessage'), TextMessage(id='226cf3e0-7c9f-47a8-a1c0-d0f4f199a871', 
                                                                                                   
                      source='A', models_usage=RequestUsage(prompt_tokens=26, completion_tokens=210), metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 2, 36, 42370, tzinfo=datetime.timezone.utc), 
                      content='Once upon a time, in a cozy little village, there was a curious gray cat named Whiskers. Every morning, Whiskers would wander outside, eager to explore the world beyond his home. One sunny day, he discovered a hidden trail in the woods that he had never seen before.\n\nIntrigued, Whiskers padded along the path, discovering vibrant flowers and chirping birds. Suddenly, he heard a soft rustling in the bushes. Out popped a tiny, shivering squirrel looking for food.\n\nFeeling compassionate, Whiskers gently nudged the squirrel with his nose. Seeing his kind gesture, the squirrel smiled and offered Whiskers a shiny acorn as a token of thanks. From that day on, Whiskers and the squirrel became the best of friends, exploring the woods and sharing adventures together.\n\nAnd so, Whiskers learned that the best treasures in life are friendship and kindness. From that day forward, he continued to explore with an open heart, always eager for new friends and new stories to tell.', type='TextMessage'), TextMessage(id='9ad33779-5256-45e2-ae36-32e9b6d29e99', 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       
                      source='C', models_usage=RequestUsage(prompt_tokens=240, completion_tokens=363), metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 2, 37, 989683, tzinfo=datetime.timezone.utc), 
                      content='昔々、居心地の良い小さな村に、好奇心旺盛な灰色の猫、ウィスカーズが住んでいました。毎朝、ウィスカーズは家の外に出て、広がる世界を探検したくてたまりませんでした。ある晴れた日、彼は森の中で今まで見たことのない隠れた小道を発見しました。\n\n興味をそそられたウィスカーズは、その道をゆっくりと歩き始め、色とりどりの花やさえずる鳥たちを見つけました。突然、茂みの中で小さな音がしました。出てきたのは、震える小さなリスで、お腹を空かせている様子でした。\n\n思いやりを感じたウィスカーズは、そっと鼻でリスを撫でました。その優しい行動に感動したリスは笑顔を見せ、感謝の印として光り輝くどんぐりをウィスカーズにプレゼントしました。その日から、ウィスカーズとリスは親友になり、一緒に森を探検し、冒険を分かち合いました。\n\nこうして、ウィスカーズは、人生で最も価値のある宝物は友情と優しさだと学びました。それ以降も、彼は心を開いて探検を続け、新しい友達や素敵な物語を見つけることにいつまでもわくわくしていました。', type='TextMessage'), TextMessage(id='bc728766-2df0-4b83-887b-f9cadb5e8d8d', 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       
                      source='B', models_usage=RequestUsage(prompt_tokens=240, completion_tokens=276), metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 2, 38, 266113, tzinfo=datetime.timezone.utc), 
                      content='很久以前，在一个温馨的小村庄里，住着一只好奇的灰色猫，名字叫胡须。每天早晨，胡须都会走出家门，渴望探索外面的世界。有一天晴朗的早晨，他在树林里发现了一条隐藏的小路，这是他以前从未见过的。\n\n出于好奇，胡须轻轻地沿着小路走着，发现了鲜艳的花朵和欢快的鸟鸣。突然，他听到灌木丛里传来轻轻的沙沙声。只见一只瘦小颤抖的松鼠从灌木中探出头，正在寻找食物。\n\n胡须看到它，心生怜悯，温柔地用鼻子轻轻推了推松鼠。松鼠看到胡须的善意微笑了，并赠送给他一颗闪亮的橡子作为感谢。从那天起，胡须和松鼠成了最好的朋友，一起探索森林，分享各种冒险。\n\n从此，胡须明白了，生命中最宝贵的财富是友谊和善良。他带着一颗开放的心继续探索，期待结识更多的朋友，讲述更多的故事。', type='TextMessage'), TextMessage(id='1b3e5db3-bb0f-455c-8793-59f21e9bc8ab', 
                                                                                                                                                                                                                                                                                                                                                                                                         
                      source='D', models_usage=RequestUsage(prompt_tokens=888, completion_tokens=143), metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 2, 39, 704547, tzinfo=datetime.timezone.utc), content='很久以前，在一个温馨的小村庄里，住着好奇的灰猫胡须。每天早晨，他都喜欢探索外面的世界。有一天，他在树林里发现了一条隐藏的小路，沿着小路，他遇见了一只瘦小颤抖的松鼠。胡须用善意的行为帮助了松鼠，作为感谢，松鼠送给他一颗闪亮的橡子。从那以后，他们成为了最好的朋友，一起探索森林，分享冒险。胡须明白，友谊和善良是生命中最宝贵的财富，并继续带着开放的心探索新朋友和故事。', type='TextMessage'), StopMessage(id='9b7ac7bc-c22a-4928-9aea-23599ae31ba6', 
                                                                                                                                                                                                                                                                                                                                                                                                                                                 
                      source='DiGraphStopAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 9, 2, 39, 706552, tzinfo=datetime.timezone.utc), 
                      content='Digraph execution is complete', type='StopMessage')] 

stop_reason='Stop message received, Maximum number of messages 5 reached, current message count: 5'

#### Example 4.

**`Conditional Branching: A → B (if ‘yes’) or C (otherwise)`**

In [52]:
import asyncio

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow
from autogen_ext.models.openai import OpenAIChatCompletionClient


# async def main():
# Initialize agents with OpenAI model clients.
model_client = OpenAIChatCompletionClient(model="gpt-4.1-nano")
agent_a = AssistantAgent(
    "A",
    model_client=model_client,
    system_message="Detect if the input is in Chinese. If it is, say 'yes', else say 'no', and nothing else.",
)
agent_b = AssistantAgent("B", model_client=model_client, system_message="Translate input to English.")
agent_c = AssistantAgent("C", model_client=model_client, system_message="Translate input to Chinese.")

# Create a directed graph with conditional branching flow A -> B ("yes"), A -> C (otherwise).
builder = DiGraphBuilder()
builder.add_node(agent_a).add_node(agent_b).add_node(agent_c)
# Create conditions as callables that check the message content.
builder.add_edge(agent_a, agent_b, condition=lambda msg: "yes" in msg.to_model_text())
builder.add_edge(agent_a, agent_c, condition=lambda msg: "yes" not in msg.to_model_text())
graph = builder.build()

# Create a GraphFlow team with the directed graph.
team = GraphFlow(
    participants=[agent_a, agent_b, agent_c],
    graph=graph,
    termination_condition=MaxMessageTermination(5),
)

# Run the team and print the events.
async for event in team.run_stream(task="AutoGen is a framework for building AI agents."):
    print(event)

# asyncio.run(main())

id='bfdb4b16-d73d-49f7-b2ed-2af72448c2c7' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 10, 31, 6, 599661, tzinfo=datetime.timezone.utc) content='AutoGen is a framework for building AI agents.' type='TextMessage'
id='d820b1c6-901f-466f-98d6-e6c65f1b1735' source='A' models_usage=RequestUsage(prompt_tokens=47, completion_tokens=1) metadata={} created_at=datetime.datetime(2025, 7, 19, 10, 31, 7, 215513, tzinfo=datetime.timezone.utc) content='no' type='TextMessage'
id='d9c18511-a83f-4421-a188-b00506c87480' source='C' models_usage=RequestUsage(prompt_tokens=33, completion_tokens=14) metadata={} created_at=datetime.datetime(2025, 7, 19, 10, 31, 7, 918259, tzinfo=datetime.timezone.utc) content='AutoGen 是一个用于构建人工智能Agent的框架。' type='TextMessage'
id='cf968bcc-f7f7-4175-b0dd-be01d887d9db' source='DiGraphStopAgent' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 10, 31, 7, 920267, tzinfo=datetime.timezone.utc) content='Digraph executio

##### Structured Output
task='AutoGen is a framework for building AI agents.'<br><br>

**`Conditional Branching: A → C (Becasue Source='A', Content='no')`**

In [ ]:
source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 10, 11, 46, 142406, tzinfo=datetime.timezone.utc) 
content='AutoGen is a framework for building AI agents.' type='TextMessage'

source='A' models_usage=RequestUsage(prompt_tokens=47, completion_tokens=1) metadata={} created_at=datetime.datetime(2025, 7, 19, 10, 11, 46, 914276, tzinfo=datetime.timezone.utc) 
content='no' type='TextMessage'

source='C' models_usage=RequestUsage(prompt_tokens=33, completion_tokens=12) metadata={} created_at=datetime.datetime(2025, 7, 19, 10, 11, 47, 491780, tzinfo=datetime.timezone.utc) 
content='AutoGen 是构建人工智能代理的框架。' type='TextMessage'

source='DiGraphStopAgent' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 10, 11, 47, 493788, tzinfo=datetime.timezone.utc) 
content='Digraph execution is complete' type='StopMessage'

messages=[TextMessage(id='89c00334-69e5-4653-a348-05d11c2939d4', 
                      
                      source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 10, 11, 46, 142406, tzinfo=datetime.timezone.utc), 
                      content='AutoGen is a framework for building AI agents.', type='TextMessage'), TextMessage(id='2c47cf36-d8b5-4db6-977a-52a6d840cccc', 
                      
                      source='A', models_usage=RequestUsage(prompt_tokens=47, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 19, 10, 11, 46, 914276, tzinfo=datetime.timezone.utc), 
                      content='no', type='TextMessage'), TextMessage(id='492d577d-a24b-4438-8456-87d477478a6d', 
                                                                     
                      source='C', models_usage=RequestUsage(prompt_tokens=33, completion_tokens=12), metadata={}, created_at=datetime.datetime(2025, 7, 19, 10, 11, 47, 491780, tzinfo=datetime.timezone.utc), 
                      content='AutoGen 是构建人工智能代理的框架。', type='TextMessage'), StopMessage(id='2bbd3166-2f8a-49a6-a4af-ba4f803b203b', 
                                                                                        
                      source='DiGraphStopAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 10, 11, 47, 493788, tzinfo=datetime.timezone.utc), 
                      content='Digraph execution is complete', type='StopMessage')] 

stop_reason='Stop message received'

In [51]:
import asyncio

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow
from autogen_ext.models.openai import OpenAIChatCompletionClient


# async def main():
# Initialize agents with OpenAI model clients.
model_client = OpenAIChatCompletionClient(model="gpt-4.1-nano")
agent_a = AssistantAgent(
    "A",
    model_client=model_client,
    system_message="Detect if the input is in Chinese. If it is, say 'yes', else say 'no', and nothing else.",
)
agent_b = AssistantAgent("B", model_client=model_client, system_message="Translate input to English.")
agent_c = AssistantAgent("C", model_client=model_client, system_message="Translate input to Chinese.")

# Create a directed graph with conditional branching flow A -> B ("yes"), A -> C (otherwise).
builder = DiGraphBuilder()
builder.add_node(agent_a).add_node(agent_b).add_node(agent_c)
# Create conditions as callables that check the message content.
builder.add_edge(agent_a, agent_b, condition=lambda msg: "yes" in msg.to_model_text())
builder.add_edge(agent_a, agent_c, condition=lambda msg: "yes" not in msg.to_model_text())
graph = builder.build()

# Create a GraphFlow team with the directed graph.
team = GraphFlow(
    participants=[agent_a, agent_b, agent_c],
    graph=graph,
    termination_condition=MaxMessageTermination(5),
)

# Run the team and print the events.
async for event in team.run_stream(task="AutoGen 是构建人工智能代理的框架。"):
    print(event)

# asyncio.run(main())

id='85daa923-e658-4895-9bc5-f702f527bb65' source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 10, 25, 52, 697283, tzinfo=datetime.timezone.utc) content='AutoGen 是构建人工智能代理的框架。' type='TextMessage'
id='200c96f6-4d21-492a-838f-fc4e312b6bf7' source='A' models_usage=RequestUsage(prompt_tokens=49, completion_tokens=1) metadata={} created_at=datetime.datetime(2025, 7, 19, 10, 25, 53, 228903, tzinfo=datetime.timezone.utc) content='yes' type='TextMessage'
id='bff472ef-fdc6-4a22-8dd7-dd89e6771478' source='B' models_usage=RequestUsage(prompt_tokens=35, completion_tokens=11) metadata={} created_at=datetime.datetime(2025, 7, 19, 10, 25, 53, 810971, tzinfo=datetime.timezone.utc) content='AutoGen is a framework for building artificial intelligence agents.' type='TextMessage'
id='186cb78c-f280-401b-8185-114b66cc166f' source='DiGraphStopAgent' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 10, 25, 53, 814208, tzinfo=datetime.timezone.utc) conten

##### Structured Output

task='AutoGen 是构建人工智能代理的框架。'<br><br>

**`Conditional Branching: A → B (Becasue Source='A', Content='yes')`**

In [ ]:
source='user' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 10, 25, 52, 697283, tzinfo=datetime.timezone.utc) 
content='AutoGen 是构建人工智能代理的框架。' type='TextMessage'

source='A' models_usage=RequestUsage(prompt_tokens=49, completion_tokens=1) metadata={} created_at=datetime.datetime(2025, 7, 19, 10, 25, 53, 228903, tzinfo=datetime.timezone.utc) 
content='yes' type='TextMessage'

source='B' models_usage=RequestUsage(prompt_tokens=35, completion_tokens=11) metadata={} created_at=datetime.datetime(2025, 7, 19, 10, 25, 53, 810971, tzinfo=datetime.timezone.utc) 
content='AutoGen is a framework for building artificial intelligence agents.' type='TextMessage'

source='DiGraphStopAgent' models_usage=None metadata={} created_at=datetime.datetime(2025, 7, 19, 10, 25, 53, 814208, tzinfo=datetime.timezone.utc) 
content='Digraph execution is complete' type='StopMessage'

messages=[TextMessage(id='85daa923-e658-4895-9bc5-f702f527bb65', 
                      
                      source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 10, 25, 52, 697283, tzinfo=datetime.timezone.utc), 
                      content='AutoGen 是构建人工智能代理的框架。', type='TextMessage'), TextMessage(id='200c96f6-4d21-492a-838f-fc4e312b6bf7', 
                                                                                        
                      source='A', models_usage=RequestUsage(prompt_tokens=49, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 19, 10, 25, 53, 228903, tzinfo=datetime.timezone.utc), 
                      content='yes', type='TextMessage'), TextMessage(id='bff472ef-fdc6-4a22-8dd7-dd89e6771478', 
                                                                      
                      source='B', models_usage=RequestUsage(prompt_tokens=35, completion_tokens=11), metadata={}, created_at=datetime.datetime(2025, 7, 19, 10, 25, 53, 810971, tzinfo=datetime.timezone.utc), 
                      content='AutoGen is a framework for building artificial intelligence agents.', type='TextMessage'), StopMessage(id='186cb78c-f280-401b-8185-114b66cc166f', 
                                                                                                                                      
                      source='DiGraphStopAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 19, 10, 25, 53, 814208, tzinfo=datetime.timezone.utc), 
                      content='Digraph execution is complete', type='StopMessage')] 

stop_reason='Stop message received'